# LIMINA -- 01. Ambil Data dari Supabase

Mengunduh enam tabel mentah dari Supabase (baca saja) dan menyimpannya sebagai
cache lokal di `data/raw/`. Notebook ini dipakai baik untuk **pengambilan data
pertama kali** (histori yang sudah terkumpul di Supabase, umumnya 90 hari atau
lebih) maupun untuk **pembaruan harian/mingguan** -- keduanya menjalankan
notebook yang sama; bedanya hanya seberapa banyak baris yang sudah tersedia di
Supabase pada saat notebook ini dijalankan.

Tidak ada satu pun operasi tulis ke Supabase di notebook ini atau di mana pun
dalam proyek ini -- seluruhnya `select()`. Lihat `limina/supabase_io.py`.

In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import pandas as pd

from limina import config, supabase_io

print(f"Root proyek: {ROOT}")

Root proyek: d:\LIMINA-model-train


## 1. Periksa kredensial Supabase

In [2]:
if not config.kredensial_lengkap():
    raise RuntimeError(
        "SUPABASE_URL dan/atau SUPABASE_KEY belum terisi.\n\n"
        "Isi lewat salah satu cara berikut, lalu jalankan ulang cell ini:\n"
        "  1. Salin .env.example menjadi .env di root proyek, isi kedua nilainya.\n"
        "  2. Atau ekspor langsung di shell sebelum menjalankan Jupyter:\n"
        "     export SUPABASE_URL='https://xxxx.supabase.co'\n"
        "     export SUPABASE_KEY='xxxx'\n\n"
        "Lihat README bagian 'Isi Kredensial Supabase' untuk detail."
    )

client = supabase_io.get_client()
assert client is not None
print("Kredensial ditemukan, klien Supabase siap.")

Kredensial ditemukan, klien Supabase siap.


## 2. Unduh enam tabel mentah (baca saja)

In [3]:
tabel_yang_diunduh = {
    config.TABEL_QUARTERLY_FINANCIALS: None,
    config.TABEL_DAILY_TRANSACTION: None,
    config.TABEL_DAILY_FULL_UNIVERSE_CLOSE: None,
    config.TABEL_FREE_FLOAT_SNAPSHOT: None,
    config.TABEL_COMPANY_OVERVIEW: None,
    config.TABEL_SUSPENSI: None,
}

for nama_tabel in tabel_yang_diunduh:
    df = supabase_io.download_table(client, nama_tabel)
    tabel_yang_diunduh[nama_tabel] = df
    print(f"{nama_tabel:32s} {len(df):>8,} baris")

quarterly_financials                   76 baris
daily_transaction                   1,185 baris
daily_full_universe_close           2,946 baris
free_float_snapshot                 2,883 baris
company_overview                       19 baris
stock_suspensions                     591 baris


## 3. Sesuaikan nama kolom tabel suspensi (kalau berbeda)

Seluruh kode di proyek ini memakai nama kolom `event_date` dan `reason` untuk tabel suspensi. Nilai bawaan `limina.config.KOLOM_TANGGAL_SUSPENSI` sudah diisi `"suspension_date"`, nama kolom sungguhan pada tabel yang sudah dikonfirmasi -- kalau tabel `stock_suspensions` Anda memakai nama lain, ganti nilai itu (dan/atau `KOLOM_ALASAN_SUSPENSI`) sekali di `limina/config.py`. Penyesuaiannya sendiri ada di `limina/supabase_io.py::normalisasi_tabel_suspensi`, diuji otomatis lewat `tests/test_supabase_io.py` -- bukan lagi kode telanjang di sel notebook.

In [4]:
df_suspensi = supabase_io.normalisasi_tabel_suspensi(tabel_yang_diunduh[config.TABEL_SUSPENSI])
tabel_yang_diunduh[config.TABEL_SUSPENSI] = df_suspensi
print(f"Kolom {config.TABEL_SUSPENSI}: {list(df_suspensi.columns)}")

Kolom stock_suspensions: ['symbol', 'event_date', 'reason', 'pdf_url', 'fetched_at', 'raw_response']


## 4. Simpan cache lokal (CSV)

In [5]:
config.RAW_DIR.mkdir(parents=True, exist_ok=True)
for nama_tabel, df in tabel_yang_diunduh.items():
    df.to_csv(config.RAW_DIR / f"{nama_tabel}.csv", index=False)

print(f"Enam tabel disimpan ke {config.RAW_DIR}")
print("Lanjut ke notebook 02 (preprocessing_dan_normalisasi) untuk membangun panel dan potret.")

Enam tabel disimpan ke D:\LIMINA-model-train\data\raw
Lanjut ke notebook 02 (preprocessing_dan_normalisasi) untuk membangun panel dan potret.
